# Production Network — Advanced Features

This notebook demonstrates the 6 advanced production network features:
1. **Artificial lift** — Gas lift, ESP, jet pump, rod pump
2. **Large-scale networks** — 100+ wells convergence
3. **Water handling** — Water cut, injection, breakthrough tracking
4. **Sand/solids transport** — Erosion rate per DNV RP O501
5. **Corrosion/integrity** — de Waard–Milliams & NORSOK M-506 models
6. **GHG emissions** — CO₂ per compressor station, field-level inventory

All features are built into `LoopedPipeNetwork`.

In [1]:
# Setup
import sys, os, time, json
import numpy as np
import matplotlib.pyplot as plt

try:
    from neqsim import jneqsim
    print('NeqSim loaded via pip package')
except ImportError:
    raise RuntimeError('Install neqsim: pip install neqsim')

LoopedPipeNetwork = jneqsim.process.equipment.network.LoopedPipeNetwork
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
print('Ready')

NeqSim loaded via pip package
Ready


In [2]:
# Helper: create gas fluid
def make_gas(p_bar=200.0):
    fluid = SystemSrkEos(273.15 + 60.0, float(p_bar))
    fluid.addComponent('methane', 0.82)
    fluid.addComponent('ethane', 0.08)
    fluid.addComponent('propane', 0.05)
    fluid.addComponent('n-butane', 0.03)
    fluid.addComponent('CO2', 0.015)
    fluid.addComponent('nitrogen', 0.005)
    fluid.setMixingRule('classic')
    return fluid

print('Helper defined')

Helper defined


## 1. Artificial Lift

Compare production rates from a 3-well network with different lift methods.

In [ ]:
def build_lift_network(lift_config=None):
    """Build a 3-well network and optionally apply artificial lift."""
    net = LoopedPipeNetwork('lift-test')
    net.setFluidTemplate(make_gas(200.0))
    net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
    net.setMaxIterations(300)
    net.setTolerance(100.0)

    net.addSourceNode('R1', 200.0, 0.0)
    net.addSourceNode('R2', 180.0, 0.0)
    net.addSourceNode('R3', 160.0, 0.0)
    net.addJunctionNode('manifold')
    net.addFixedPressureSinkNode('export', 60.0)

    net.addWellIPR('R1', 'manifold', 'W1', 5e-13, True)
    net.addWellIPR('R2', 'manifold', 'W2', 5e-13, True)
    net.addWellIPR('R3', 'manifold', 'W3', 5e-13, True)
    net.addPipe('manifold', 'export', 'trunk', 20000.0, 0.3)

    if lift_config:
        for well, cfg in lift_config.items():
            if cfg['type'] == 'gas_lift':
                net.setGasLift(well, float(cfg['rate']))
            elif cfg['type'] == 'esp':
                net.setESP(well, float(cfg['power']), float(cfg['eff']))

    net.run()
    return net

# Run cases
cases = {
    'No lift': None,
    'GL on W3 (500 kg/hr)': {'W3': {'type': 'gas_lift', 'rate': 500}},
    'ESP on W3 (80 kW)': {'W3': {'type': 'esp', 'power': 80, 'eff': 0.55}},
    'GL + ESP (W2+W3)': {
        'W2': {'type': 'gas_lift', 'rate': 300},
        'W3': {'type': 'esp', 'power': 60, 'eff': 0.50}
    },
}

results = {}
for name, cfg in cases.items():
    net = build_lift_network(cfg)
    total_flow = abs(float(net.getTotalSinkFlow())) * 3600  # kg/hr
    gl_rate = float(net.getTotalGasLiftRate())
    esp_power = float(net.getTotalESPPower())
    results[name] = {'flow_kghr': total_flow, 'gl_kghr': gl_rate, 'esp_kW': esp_power}
    print(f'{name:30s}  Flow: {total_flow:10.1f} kg/hr  GL: {gl_rate:.0f} kg/hr  ESP: {esp_power:.0f} kW')

TypeError: No matching overloads found for neqsim.process.equipment.network.LoopedPipeNetwork.addJunctionNode(str,float), options are:
	public void neqsim.process.equipment.network.LoopedPipeNetwork.addJunctionNode(java.lang.String)


In [ ]:
# Plot lift comparison
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
flows = [results[n]['flow_kghr'] for n in names]
colors = ['#607D8B', '#4CAF50', '#2196F3', '#FF9800']
bars = ax.barh(names, flows, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Total Production (kg/hr)', fontsize=12)
ax.set_title('Artificial Lift Impact on Production Rate', fontsize=14)
ax.axvline(flows[0], color='red', linestyle='--', alpha=0.5, label='Base (no lift)')
for bar, flow in zip(bars, flows):
    pct = (flow / flows[0] - 1) * 100
    ax.text(bar.get_width() + max(flows)*0.01, bar.get_y() + bar.get_height()/2,
            f'{pct:+.1f}%', va='center', fontsize=10)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Large-Scale Network (120 Wells)

Demonstrate convergence of a network with 120 wells across 6 manifolds.

In [ ]:
net = LoopedPipeNetwork('large-scale')
net.setFluidTemplate(make_gas(250.0))
net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
net.setMaxIterations(500)
net.setTolerance(500.0)

# 6 manifolds, 20 wells each
for m in range(6):
    mf_name = f'MF{m+1}'
    net.addJunctionNode(mf_name)
    for w in range(20):
        res_p = 230.0 + np.random.uniform(-30, 30)
        pi = 5e-13 * np.random.uniform(0.5, 2.0)
        src = f'R{m}_{w}'
        well = f'W{m}_{w}'
        net.addSourceNode(src, float(res_p), 0.0)
        net.addWellIPR(src, mf_name, well, float(pi), True)

# Connect manifolds to central hub
net.addJunctionNode('hub')
for m in range(6):
    dist = 5000.0 + m * 3000.0
    net.addPipe(f'MF{m+1}', 'hub', f'trunk{m+1}', float(dist), 0.3)

net.addFixedPressureSinkNode('export', 70.0)
net.addPipe('hub', 'export', 'export_line', 30000.0, 0.4)

t0 = time.time()
net.run()
elapsed = time.time() - t0

print(f'Network: 120 wells, 6 manifolds')
print(f'Converged: {net.isConverged()}')
print(f'Iterations: {net.getIterationCount()}')
print(f'Solve time: {elapsed:.2f} s')
print(f'Total production: {abs(float(net.getTotalSinkFlow()))*3600:.0f} kg/hr')

## 3. Water Handling

Track water cuts, compute water balance, and simulate water breakthrough over time.

In [ ]:
# Build production network
net = LoopedPipeNetwork('water-handling')
net.setFluidTemplate(make_gas(200.0))
net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
net.setMaxIterations(200)
net.setTolerance(100.0)

net.addSourceNode('R1', 200.0, 0.0)
net.addSourceNode('R2', 190.0, 0.0)
net.addSourceNode('R3', 180.0, 0.0)
net.addJunctionNode('manifold')
net.addFixedPressureSinkNode('export', 60.0)

net.addWellIPR('R1', 'manifold', 'W1', 5e-13, True)
net.addWellIPR('R2', 'manifold', 'W2', 5e-13, True)
net.addWellIPR('R3', 'manifold', 'W3', 5e-13, True)
net.addPipe('manifold', 'export', 'trunk', 15000.0, 0.3)

# Set initial water cuts
net.setWaterCut('W1', 0.10)
net.setWaterCut('W2', 0.30)
net.setWaterCut('W3', 0.05)

# Add water injection
net.addWaterInjection('W2', 2000.0)  # 2000 kg/hr injection support

# Water breakthrough tracking
net.setWaterBreakthrough('W1', 365.0, 0.10, 0.65)  # day, initial WC, final WC
net.setWaterBreakthrough('W2', 180.0, 0.30, 0.80)

net.run()

# Calculate water balance
wb = net.calculateWaterBalance()

print(f'Converged: {net.isConverged()}')
print(f'Total water production:  {float(net.getTotalWaterProduction()):.1f} kg/hr')
print(f'Total water injection:   {float(net.getTotalWaterInjection()):.1f} kg/hr')
print(f'\nWater balance per pipe:')
for name in wb.keySet():
    vals = wb.get(name)
    wc = float(vals[0])
    wf = float(vals[1])
    inj = float(vals[2])
    print(f'  {name:10s}  WC={wc:.1%}  WaterFlow={wf:.1f} kg/hr  Injection={inj:.1f} kg/hr')

In [ ]:
# Plot water breakthrough profile for W1 and W2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# W1: breakthrough at day 365, 10% -> 65%
days = np.linspace(0, 3*365, 200)
wc_w1 = np.where(days < 365, 0.10, 0.10 + (0.65 - 0.10) * (1 - np.exp(-0.003 * (days - 365))))
wc_w2 = np.where(days < 180, 0.30, 0.30 + (0.80 - 0.30) * (1 - np.exp(-0.003 * (days - 180))))

ax1.plot(days/365, wc_w1*100, 'b-', linewidth=2, label='W1 (BT day 365)')
ax1.plot(days/365, wc_w2*100, 'r-', linewidth=2, label='W2 (BT day 180)')
ax1.set_xlabel('Time (years)', fontsize=12)
ax1.set_ylabel('Water Cut (%)', fontsize=12)
ax1.set_title('Water Breakthrough Profiles', fontsize=14)
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 100)

# Water balance bar chart
wells = ['W1', 'W2', 'W3']
water_cuts = [10, 30, 5]
ax2.bar(wells, water_cuts, color=['#2196F3', '#F44336', '#4CAF50'], edgecolor='black')
ax2.set_ylabel('Water Cut (%)', fontsize=12)
ax2.set_title('Current Water Cut per Well', fontsize=14)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Sand Transport & Erosion (DNV RP O501)

Calculate erosion rates and check against design limits.

In [ ]:
net = LoopedPipeNetwork('sand-test')
net.setFluidTemplate(make_gas(200.0))
net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
net.setMaxIterations(200)
net.setTolerance(100.0)

net.addSourceNode('R1', 200.0, 0.0)
net.addSourceNode('R2', 180.0, 0.0)
net.addJunctionNode('manifold')
net.addFixedPressureSinkNode('export', 60.0)

net.addWellIPR('R1', 'manifold', 'W1', 5e-13, True)
net.addWellIPR('R2', 'manifold', 'W2', 5e-13, True)
net.addPipe('manifold', 'export', 'trunk', 15000.0, 0.3)

# Set sand rates (kg/hr)
net.setSandRate('W1', 3.0)
net.setSandRate('W2', 12.0)  # Higher sand producer
net.setMaxAllowableSandRate(10.0)  # kg/hr limit
net.setMaxAllowableErosionRate(5.0)  # mm/yr limit

net.run()

# Calculate sand transport per DNV RP O501
sand = net.calculateSandTransport()

print(f'Sand Transport Analysis (DNV RP O501)')
print(f'={"":-<60}')
for name in sand.keySet():
    vals = sand.get(name)
    rate = float(vals[0])
    conc = float(vals[1])
    erosion = float(vals[2])
    depo = float(vals[3])
    print(f'  {name:10s}  Sand={rate:.1f} kg/hr  Conc={conc:.4f} kg/m3  '
          f'Erosion={erosion:.2f} mm/yr  Deposition={depo:.2f} mm/yr')

# Check violations
violations = list(net.getSandViolations())
print(f'\nViolations: {len(violations)}')
for v in violations:
    print(f'  ⚠ {v}')

In [ ]:
# Plot sand sensitivity: erosion rate vs sand production rate
sand_rates = np.linspace(0, 30, 50)
erosion_rates = []

for sr in sand_rates:
    n = LoopedPipeNetwork('sand-sweep')
    n.setFluidTemplate(make_gas(200.0))
    n.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
    n.setMaxIterations(200)
    n.setTolerance(100.0)
    n.addSourceNode('R', 200.0, 0.0)
    n.addJunctionNode('j')
    n.addFixedPressureSinkNode('exp', 60.0)
    n.addWellIPR('R', 'j', 'W', 5e-13, True)
    n.addPipe('j', 'exp', 'pipe', 10000.0, 0.2)
    n.setSandRate('W', float(sr))
    n.run()
    res = n.calculateSandTransport()
    # Get max erosion from all pipes
    max_e = 0.0
    for name in res.keySet():
        e = float(res.get(name)[2])
        if e > max_e:
            max_e = e
    erosion_rates.append(max_e)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sand_rates, erosion_rates, 'b-', linewidth=2)
ax.axhline(5.0, color='red', linestyle='--', linewidth=1.5, label='Max allowable (5 mm/yr)')
ax.fill_between(sand_rates, 5.0, max(max(erosion_rates), 6), alpha=0.1, color='red')
ax.set_xlabel('Sand Production Rate (kg/hr)', fontsize=12)
ax.set_ylabel('Erosion Rate (mm/yr)', fontsize=12)
ax.set_title('Sand Erosion Sensitivity (DNV RP O501)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Corrosion Assessment

Compare de Waard–Milliams (1975) and NORSOK M-506 (2005) corrosion models.

In [ ]:
def run_corrosion_test(co2_frac, h2s_frac, model='deWaard'):
    """Run a network and return corrosion results for a single pipe."""
    net = LoopedPipeNetwork('corr-test')
    net.setFluidTemplate(make_gas(200.0))
    net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
    net.setMaxIterations(200)
    net.setTolerance(100.0)

    net.addSourceNode('R', 200.0, 0.0)
    net.addJunctionNode('j')
    net.addFixedPressureSinkNode('exp', 60.0)
    net.addWellIPR('R', 'j', 'W', 5e-13, True)
    net.addPipe('j', 'exp', 'pipe', 10000.0, 0.2)

    net.setCorrosiveGas('pipe', float(co2_frac), float(h2s_frac))
    net.setCorrosionModel('pipe', str(model))
    net.setMinAllowableWallLife(20.0)

    net.run()
    corr = net.calculateCorrosion()
    if corr.containsKey('pipe'):
        vals = corr.get('pipe')
        return float(vals[0])  # corrosion rate mm/yr
    return 0.0

# Sweep CO2 mole fraction
co2_fracs = np.linspace(0.005, 0.10, 30)
dw_rates = [run_corrosion_test(co2, 0.001, 'deWaard') for co2 in co2_fracs]
norsok_rates = [run_corrosion_test(co2, 0.001, 'norsokM506') for co2 in co2_fracs]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(co2_fracs * 100, dw_rates, 'b-o', markersize=4, linewidth=2, label='de Waard–Milliams')
ax.plot(co2_fracs * 100, norsok_rates, 'r-s', markersize=4, linewidth=2, label='NORSOK M-506')
ax.set_xlabel('CO₂ Mole Fraction (%)', fontsize=12)
ax.set_ylabel('Corrosion Rate (mm/yr)', fontsize=12)
ax.set_title('CO₂ Corrosion Rate Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed corrosion report for a production network
net = LoopedPipeNetwork('corr-detail')
net.setFluidTemplate(make_gas(200.0))
net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
net.setMaxIterations(200)
net.setTolerance(100.0)

net.addSourceNode('R1', 200.0, 0.0)
net.addSourceNode('R2', 180.0, 0.0)
net.addJunctionNode('manifold')
net.addFixedPressureSinkNode('export', 60.0)

net.addWellIPR('R1', 'manifold', 'W1', 5e-13, True)
net.addWellIPR('R2', 'manifold', 'W2', 5e-13, True)
net.addPipe('manifold', 'export', 'trunk', 20000.0, 0.3)

# Different corrosive environments
net.setCorrosiveGas('W1', 0.03, 0.001)    # 3% CO2, 0.1% H2S
net.setCorrosiveGas('W2', 0.06, 0.005)    # 6% CO2, 0.5% H2S
net.setCorrosiveGas('trunk', 0.04, 0.002) # Mixed at manifold

net.setMinAllowableWallLife(20.0)  # years

net.run()
corr = net.calculateCorrosion()

print(f'Corrosion Assessment (de Waard–Milliams)')
print(f'={"":-<70}')
for name in corr.keySet():
    vals = corr.get(name)
    rate = float(vals[0])
    pco2 = float(vals[1]) if len(vals) > 1 else 0
    life = float(vals[2]) if len(vals) > 2 else 0
    print(f'  {name:10s}  Rate={rate:.3f} mm/yr  pCO2={pco2:.2f} bar  '
          f'RemLife={life:.1f} yr')

violations = list(net.getCorrosionViolations())
if violations:
    print(f'\n⚠ Integrity violations ({len(violations)}):')
    for v in violations:
        print(f'  {v}')
else:
    print('\n✓ All pipes within integrity limits')

## 6. GHG Emissions Inventory

Calculate CO₂ and methane emissions per compressor station with field-level totals.

In [ ]:
# Build network with compressor
net = LoopedPipeNetwork('emissions')
net.setFluidTemplate(make_gas(200.0))
net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
net.setMaxIterations(200)
net.setTolerance(100.0)

net.addSourceNode('field', 200.0, 0.0)
net.addJunctionNode('j1')
net.addJunctionNode('j2')
net.addFixedPressureSinkNode('delivery', 70.0)

net.addPipe('field', 'j1', 'inflow', 5000, 0.3)
net.addCompressor('j1', 'j2', 'booster', 0.75)
net.addPipe('j2', 'delivery', 'export', 10000, 0.3)

net.setCO2EmissionFactor(2.75)     # kg CO2 per kg fuel gas
net.setMethaneSlipFactor(0.02)     # 2% methane slip

net.run()
print(f'Converged: {net.isConverged()}')

# Set compressor power (known operating scenario for demo)
comp = net.getPipe('booster')
comp.setCompressorPower(1500.0)  # 1500 kW gas turbine driver

emissions = net.calculateEmissions()

print(f'\nGHG Emissions Inventory')
print(f'={"":-<60}')
for name in emissions.keySet():
    em = emissions.get(name)
    co2 = float(em[0])
    ch4 = float(em[1])
    co2eq = float(em[2])
    power = float(em[3])
    fuel = float(em[4])
    print(f'  Compressor: {name}')
    print(f'    Power:         {power:10.1f} kW')
    print(f'    Fuel gas:      {fuel:10.1f} kg/hr')
    print(f'    CO₂ direct:    {co2:10.1f} kg/hr')
    print(f'    CH₄ slip:      {ch4:10.3f} kg/hr')
    print(f'    CO₂-eq:        {co2eq:10.1f} kg/hr')

print(f'\nField Totals:')
print(f'  Total CO₂-eq:     {float(net.getTotalCO2Emissions()):10.1f} kg/hr')
print(f'  Annual CO₂-eq:    {float(net.getAnnualCO2EmissionsTonnes()):10.0f} tonnes/yr')
total_prod = abs(float(net.getTotalSinkFlow())) * 3600
if total_prod > 0:
    intensity = float(net.getEmissionsIntensity())
    print(f'  Intensity:        {intensity:10.1f} kgCO₂-eq/tonne')

In [ ]:
# Power sensitivity: emissions vs compressor power
powers = np.linspace(100, 5000, 40)
annual_co2 = []

for p in powers:
    n = LoopedPipeNetwork('em-sweep')
    n.setFluidTemplate(make_gas(200.0))
    n.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
    n.setMaxIterations(200)
    n.setTolerance(100.0)
    n.addSourceNode('S', 200.0, 0.0)
    n.addJunctionNode('a')
    n.addJunctionNode('b')
    n.addFixedPressureSinkNode('D', 70.0)
    n.addPipe('S', 'a', 'p1', 5000, 0.3)
    n.addCompressor('a', 'b', 'C', 0.75)
    n.addPipe('b', 'D', 'p2', 5000, 0.3)
    n.setCO2EmissionFactor(2.75)
    n.setMethaneSlipFactor(0.02)
    n.run()
    n.getPipe('C').setCompressorPower(float(p))
    n.calculateEmissions()
    annual_co2.append(float(n.getAnnualCO2EmissionsTonnes()))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(powers / 1000, np.array(annual_co2) / 1000, 'g-', linewidth=2)
ax.set_xlabel('Compressor Power (MW)', fontsize=12)
ax.set_ylabel('Annual CO₂-eq Emissions (ktonnes/yr)', fontsize=12)
ax.set_title('GHG Emissions vs Compression Power', fontsize=14)
ax.grid(alpha=0.3)

# Add EU ETS reference price annotation
ax2 = ax.twinx()
ets_price = 80  # EUR/tonne
carbon_cost = np.array(annual_co2) * ets_price / 1e6  # MEUR/yr
ax2.plot(powers / 1000, carbon_cost, 'r--', linewidth=1.5, alpha=0.7)
ax2.set_ylabel('Carbon Cost @ €80/t (MEUR/yr)', fontsize=11, color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

## Combined Feature Demo

All 6 features used together in a single production network simulation.

In [ ]:
# Build integrated production network
net = LoopedPipeNetwork('integrated')
net.setFluidTemplate(make_gas(250.0))
net.setSolverType(LoopedPipeNetwork.SolverType.NEWTON_RAPHSON)
net.setMaxIterations(300)
net.setTolerance(200.0)

# 4 wells at different reservoir pressures
for i, (rp, pi) in enumerate([(250, 5e-13), (220, 4e-13), (200, 6e-13), (180, 3e-13)]):
    net.addSourceNode(f'R{i+1}', float(rp), 0.0)
    net.addWellIPR(f'R{i+1}', 'manifold', f'W{i+1}', float(pi), True)

net.addJunctionNode('manifold')
net.addJunctionNode('j2')
net.addFixedPressureSinkNode('export', 60.0)
net.addCompressor('manifold', 'j2', 'comp', 0.75)
net.addPipe('j2', 'export', 'trunk', 25000.0, 0.3)

# Feature 1: Artificial lift
net.setGasLift('W3', 400.0)        # Gas lift on well 3
net.setESP('W4', 60.0, 0.50)       # ESP on well 4

# Feature 3: Water cuts
net.setWaterCut('W1', 0.15)
net.setWaterCut('W2', 0.40)
net.setWaterCut('W3', 0.25)
net.setWaterCut('W4', 0.08)

# Feature 4: Sand
net.setSandRate('W1', 2.0)
net.setSandRate('W2', 10.0)
net.setSandRate('W3', 5.0)
net.setMaxAllowableErosionRate(5.0)

# Feature 5: Corrosion
net.setCorrosiveGas('trunk', 0.035, 0.002)
net.setMinAllowableWallLife(20.0)

# Feature 6: Emissions
net.setCO2EmissionFactor(2.75)
net.setMethaneSlipFactor(0.02)

# Solve
net.run()
print(f'Converged: {net.isConverged()}')
print(f'Total production: {abs(float(net.getTotalSinkFlow()))*3600:.0f} kg/hr')

# Set compressor power for emissions
net.getPipe('comp').setCompressorPower(800.0)

# Run all post-processing
net.calculateWaterBalance()
net.calculateSandTransport()
net.calculateCorrosion()
net.calculateEmissions()

# Summary
print(f'\n--- Post-Processing Summary ---')
print(f'Gas lift total:     {float(net.getTotalGasLiftRate()):.0f} kg/hr')
print(f'ESP total power:    {float(net.getTotalESPPower()):.0f} kW')
print(f'Water production:   {float(net.getTotalWaterProduction()):.0f} kg/hr')
print(f'Sand violations:    {len(list(net.getSandViolations()))}')
print(f'Corrosion violations: {len(list(net.getCorrosionViolations()))}')
print(f'Annual CO₂-eq:      {float(net.getAnnualCO2EmissionsTonnes()):.0f} tonnes/yr')

In [ ]:
# Print full network report
report = str(net.getNetworkReport())
print(report[:3000])  # First 3000 chars

In [ ]:
# Parse JSON output
json_str = str(net.toJson())
data = json.loads(json_str)
print(f'JSON keys: {list(data.keys())}')
if 'emissions' in data:
    print(f'Emissions: {json.dumps(data["emissions"], indent=2)}')
if 'waterBalance' in data:
    print(f'Water balance: {json.dumps(data["waterBalance"], indent=2)}')

## Summary

| Feature | Method | Standard/Model |
|---------|--------|----------------|
| Artificial lift | `setGasLift()`, `setESP()`, `setJetPump()`, `setRodPump()` | Gas-lift GLR model, ESP power-efficiency |
| Large-scale | Standard NR-GGA solver | Todini-Pilati 1988 |
| Water handling | `setWaterCut()`, `addWaterInjection()`, `calculateWaterBalance()` | Mass balance tracking |
| Sand/erosion | `setSandRate()`, `calculateSandTransport()` | **DNV RP O501** |
| Corrosion | `setCorrosiveGas()`, `calculateCorrosion()` | **de Waard–Milliams**, **NORSOK M-506** |
| Emissions | `calculateEmissions()`, `getAnnualCO2EmissionsTonnes()` | IPCC AR5 GWP, fuel gas combustion |